In [4]:
# 初始化环境：安装依赖
%pip install -r ../requirements.txt -i https://pypi.tuna.tsinghua.edu.cn/simple

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simpleNote: you may need to restart the kernel to use updated packages.



# 实验三 步骤一：数据集划分

本步骤将 lab2 中的 PR 数据按照比例划分为训练集、验证集和测试集。

- **输出**：三个 JSON 文件 `data/train.json`, `data/val.json`, `data/test.json`
- **每个样本字段**：`pr_id`, `repo`, `before_path`, `after_path`, `merged`, `label`
- **label**：1 = PR 已合并，0 = PR 未合并
- **路径**：`before_path` 和 `after_path` 是相对于 lab2 根目录的相对路径

In [5]:
import sys
import os
sys.path.append(os.getcwd())

In [6]:
# 重新加载 config（避免 pycache 缓存旧版本）
import importlib
import sys
sys.path.insert(0, "..")
import config
importlib.reload(config)

# 导入划分模块
from dataset_splitter import DatasetSplitter

In [7]:
# 创建划分器并运行
splitter = DatasetSplitter()
result = splitter.run()

开始数据集划分...

处理仓库: kubernetes/kubernetes
  处理 295 个 PR...

处理仓库: pytorch/pytorch
  处理 210 个 PR...

处理仓库: tensorflow/tensorflow
  处理 11 个 PR...

处理仓库: microsoft/vscode
  处理 104 个 PR...

处理仓库: langchain-ai/langchain
  处理 135 个 PR...
数据集划分完成
  总有效样本: 687
    已合并(label=1): 190
    未合并(label=0): 497
    跳过(无代码): 68
------------------------------------------------------------
  训练集 (80%): 549 样本
    已合并: 145, 未合并: 404
  验证集 (10%): 68 样本
    已合并: 24, 未合并: 44
  测试集 (10%): 70 样本
    已合并: 21, 未合并: 49
------------------------------------------------------------
  保存路径:
    f:\学习资料\智能软件工程实践\lab3\data\train.json
    f:\学习资料\智能软件工程实践\lab3\data\val.json
    f:\学习资料\智能软件工程实践\lab3\data\test.json


## 检查划分结果

可以加载保存的 JSON 文件查看内容格式

In [8]:
import json
import sys
sys.path.insert(0, "..")
import config

# 读取训练集
with open(config.TRAIN_JSON_PATH, 'r', encoding='utf-8') as f:
    train_data = json.load(f)

print(f'训练集样本数: {len(train_data)}')
if len(train_data) > 0:
    print('第一个样本格式:')
    print(json.dumps(train_data[0], indent=2, ensure_ascii=False))

训练集样本数: 549
第一个样本格式:
{
  "pr_id": 140061,
  "repo": "kubernetes_kubernetes",
  "before_path": "data\\code\\human\\kubernetes_kubernetes\\140061\\before",
  "after_path": "data\\code\\human\\kubernetes_kubernetes\\140061\\after",
  "merged": false,
  "label": 0
}


In [9]:
# 读取验证集
with open(config.VAL_JSON_PATH, 'r', encoding='utf-8') as f:
    val_data = json.load(f)
    
print(f'验证集样本数: {len(val_data)}')

验证集样本数: 68


In [10]:
# 读取测试集
with open(config.TEST_JSON_PATH, 'r', encoding='utf-8') as f:
    test_data = json.load(f)
    
print(f'测试集样本数: {len(test_data)}')

测试集样本数: 70


---
## 步骤二：Code2Vec 代码向量化

使用 tree-sitter 从代码中提取 AST 路径，通过 Code2Vec 模型将 before/after 代码转化为向量。

- **流程**：构建词汇表 → 创建 Code2Vec 模型 → 向量化三个数据集
- **输出**：`data/vectors/vocab.json`, `data/vectors/train_vectors.pt`, `val_vectors.pt`, `test_vectors.pt`
- **向量维度**：128 维

In [11]:
# 导入向量化模块
import importlib
import sys
sys.path.insert(0, "..")
import config
importlib.reload(config)

from vectorize_code import CodeVectorizer

In [12]:
# 创建向量化器并运行
# 设为 True 跳过向量化（如果已生成过），设为 False 重新生成
SKIP_VECTORIZE = False

vectorizer = CodeVectorizer()
if not SKIP_VECTORIZE:
    vectorizer.run()
else:
    print("SKIP_VECTORIZE=True，跳过向量化步骤")

使用设备: cuda
开始 Code2Vec 代码向量化...

所有向量文件已存在，跳过全部向量化:
  f:\学习资料\智能软件工程实践\lab3\data\vectors\train_vectors.pt
  f:\学习资料\智能软件工程实践\lab3\data\vectors\val_vectors.pt
  f:\学习资料\智能软件工程实践\lab3\data\vectors\test_vectors.pt

如需重新生成，请设置 force=True 或手动删除上述文件。


## 检查向量化结果

加载保存的 .pt 文件，查看向量形状和样本格式

In [13]:
import torch
import sys
sys.path.insert(0, "..")
import config

# 加载训练集向量
train_data = torch.load(config.TRAIN_VECTORS_PATH, weights_only=False)
print(f'训练集向量:')
print(f'  PR 数量: {len(train_data["pr_ids"])}')
print(f'  before_vectors 形状: {train_data["before_vectors"].shape}')
print(f'  after_vectors 形状: {train_data["after_vectors"].shape}')
print(f'  labels 分布: {train_data["labels"].sum().item()} 个正样本(已合并)')
print()

val_data = torch.load(config.VAL_VECTORS_PATH, weights_only=False)
print(f'验证集向量:')
print(f'  PR 数量: {len(val_data["pr_ids"])}')
print(f'  before_vectors 形状: {val_data["before_vectors"].shape}')
print()

test_data = torch.load(config.TEST_VECTORS_PATH, weights_only=False)
print(f'测试集向量:')
print(f'  PR 数量: {len(test_data["pr_ids"])}')
print(f'  before_vectors 形状: {test_data["before_vectors"].shape}')

训练集向量:
  PR 数量: 549
  before_vectors 形状: torch.Size([549, 128])
  after_vectors 形状: torch.Size([549, 128])
  labels 分布: 145 个正样本(已合并)

验证集向量:
  PR 数量: 68
  before_vectors 形状: torch.Size([68, 128])

测试集向量:
  PR 数量: 70
  before_vectors 形状: torch.Size([70, 128])


---
## 步骤三：分类器训练与评估

将 before/after 向量拼接，分别用随机森林、SVM、MLP 训练分类器，
评估模型在测试集上的性能，并绘制可视化图表。

- **向量组合**：`[before_vec, after_vec]` 拼接 → 256 维
- **分类器**：RandomForest, SVM, MLP
- **评估指标**：Accuracy, Precision, Recall, F1-score, ROC-AUC

In [14]:
# 导入模块
import importlib
import sys
sys.path.insert(0, "..")
import config
importlib.reload(config)

import numpy as np
import torch

from train_random_forest import RandomForestTrainer
from train_svm import SVMTrainer
from train_mlp import MLPTrainer
from evaluator import Evaluator

In [15]:
# 加载向量数据并拼接
train_data = torch.load(config.TRAIN_VECTORS_PATH, weights_only=False)
test_data = torch.load(config.TEST_VECTORS_PATH, weights_only=False)

X_train = torch.cat([train_data["before_vectors"], train_data["after_vectors"]], dim=1).numpy()
y_train = train_data["labels"].numpy()

X_test = torch.cat([test_data["before_vectors"], test_data["after_vectors"]], dim=1).numpy()
y_test = test_data["labels"].numpy()

print(f"训练集: X={X_train.shape}, y={y_train.shape}")
print(f"测试集: X={X_test.shape}, y={y_test.shape}")
print(f"正样本(已合并): 训练集={y_train.sum()}, 测试集={y_test.sum()}")

训练集: X=(549, 256), y=(549,)
测试集: X=(70, 256), y=(70,)
正样本(已合并): 训练集=145, 测试集=21


In [16]:
# 训练开关：True=跳过训练（加载已有模型），False=重新训练
SKIP_TRAIN = False

In [17]:
# 初始化评估器
evaluator = Evaluator()
evaluator.set_test_data(y_test)

### 1. 随机森林

In [18]:
print("训练随机森林...")
rf_trainer = RandomForestTrainer()
if SKIP_TRAIN and os.path.exists(rf_trainer.model_path):
    rf_trainer.load()
else:
    rf_trainer.train(X_train, y_train)
    rf_trainer.save()

y_pred_rf = rf_trainer.predict(X_test)
y_proba_rf = rf_trainer.predict_proba(X_test)
evaluator.add_model("RandomForest", y_pred_rf, y_proba_rf)

训练随机森林...
  训练集准确率: 0.9982
  模型已保存到: f:\学习资料\智能软件工程实践\lab3\data\results\models\random_forest.pkl


### 2. SVM

In [19]:
print("训练 SVM...")
svm_trainer = SVMTrainer()
if SKIP_TRAIN and os.path.exists(svm_trainer.model_path):
    svm_trainer.load()
else:
    svm_trainer.train(X_train, y_train)
    svm_trainer.save()

y_pred_svm = svm_trainer.predict(X_test)
y_proba_svm = svm_trainer.predict_proba(X_test)
evaluator.add_model("SVM", y_pred_svm, y_proba_svm)

训练 SVM...
  训练集准确率: 0.7760
  模型已保存到: f:\学习资料\智能软件工程实践\lab3\data\results\models\svm.pkl


### 3. MLP

In [20]:
print("训练 MLP...")
mlp_trainer = MLPTrainer()
if SKIP_TRAIN and os.path.exists(mlp_trainer.model_path):
    mlp_trainer.load()
else:
    mlp_trainer.train(X_train, y_train)
    mlp_trainer.save()

y_pred_mlp = mlp_trainer.predict(X_test)
y_proba_mlp = mlp_trainer.predict_proba(X_test)
evaluator.add_model("MLP", y_pred_mlp, y_proba_mlp)

训练 MLP...
  训练集准确率: 0.8106
  迭代次数: 17
  模型已保存到: f:\学习资料\智能软件工程实践\lab3\data\results\models\mlp.pkl


## 评估结果

In [21]:
# 计算指标并打印汇总
evaluator.compute_all()
evaluator.print_summary()
evaluator.save_results()


评估结果汇总
Model               Accuracy Precision    Recall        F1   Roc_auc
--------------------------------------------------------------------
RandomForest          0.7714    1.0000    0.2381    0.3846    0.7677
SVM                   0.7143    0.5161    0.7619    0.6154    0.7775
MLP                   0.7571    0.8333    0.2381    0.3704    0.7279
评估结果已保存: f:\学习资料\智能软件工程实践\lab3\data\results\classification_results.json


## 可视化图表

In [22]:
# 混淆矩阵
evaluator.plot_confusion_matrices_all()

  混淆矩阵已保存: f:\学习资料\智能软件工程实践\lab3\image\confusion_matrix_RandomForest.png
  混淆矩阵已保存: f:\学习资料\智能软件工程实践\lab3\image\confusion_matrix_SVM.png
  混淆矩阵已保存: f:\学习资料\智能软件工程实践\lab3\image\confusion_matrix_MLP.png


In [23]:
# ROC 曲线对比
evaluator.plot_roc_curves()

  ROC 曲线已保存: f:\学习资料\智能软件工程实践\lab3\image\roc_curves.png


In [24]:
# 指标对比柱状图
evaluator.plot_metrics_comparison()

  指标对比图已保存: f:\学习资料\智能软件工程实践\lab3\image\metrics_comparison.png


In [25]:
# 随机森林特征重要性
evaluator.plot_feature_importance(rf_trainer.get_feature_importance())

  特征重要性已保存: f:\学习资料\智能软件工程实践\lab3\image\feature_importance.png


In [26]:
# MLP 训练损失曲线
evaluator.plot_loss_curve(mlp_trainer.get_loss_curve())

  损失曲线已保存: f:\学习资料\智能软件工程实践\lab3\image\mlp_loss_curve.png
